In [105]:
%run BuildGraph.ipynb

[[7, 0], [1, 0], [4, 0], [10, 7], [12, 10], [25, 18], [20, 18], [24, 25], [10, 5], [9, 5], [14, 12], [21, 19], [22, 21], [23, 21], [17, 22], [23, 22], [18, 17], [27, 20], [8, 3], [5, 3], [11, 8], [9, 8], [19, 11], [2, 1], [3, 2], [6, 2], [13, 9], [17, 13], [19, 13], [29, 27], [31, 27], [32, 29], [30, 24], [29, 24], [33, 30], [32, 30], [36, 33], [32, 31], [34, 31], [33, 28], [25, 28], [26, 23], [36, 26], [28, 26], [35, 34], [11, 6], [7, 4], [12, 4], [16, 14], [15, 14], [15, 16], [20, 15]]
(29, 32, {'weight': 0.7, 'required': 1})
(30, 32, {'weight': 1.3, 'required': 1})
(32, 29, {'weight': 0.4, 'required': 0})
(32, 30, {'weight': 0.4, 'required': 0})
(32, 31, {'weight': 0.4, 'required': 0})
(31, 32, {'weight': 1.1, 'required': 1})


In [106]:
from gurobipy import *

# Create a new model
m = Model()

In [107]:
def edgeName(u,v,reversed = False):
    if(reversed):
        return str(v) + "_" + str(u) if u < v else str(u) + "_" + str(v)
    else:
        return str(u) + "_" + str(v) if u < v else str(v) + "_" + str(u)


In [108]:
#for each edge, create a variable in the model
gurobiVars = dict()
obj = LinExpr()

for u,v,a in G.edges(data=True):
    thisEdgeName = str(u) + "_" + str(v)
    thisVar = m.addVar(vtype=GRB.INTEGER, name=thisEdgeName)
    gurobiVars[thisEdgeName] = thisVar
    obj += (a['weight'] * (thisVar))
    
m.setObjective(obj, GRB.MINIMIZE)

In [109]:
#Constraints: all required trails must be hiked in at least one of their two directions
for u,v,a in G.edges(data=True):
    if(a['required']==1):
        constraintForThisEdge = LinExpr()
        
        constraintForThisEdge += gurobiVars[str(u) + "_" + str(v)]
        constraintForThisEdge += gurobiVars[str(v) + "_" + str(u)]

        m.addConstr(constraintForThisEdge >= 1)
    

In [110]:
#Tour - for all nodes, in-degree must match out degree
for n in G.nodes():
    inDegreeThisNode = LinExpr()
    outDegreeThisNode = LinExpr()
    for u,v,a in G.edges(n, data=True):
        #is this edge going into our node or out?
        outDegreeThisNode += gurobiVars[str(u) + "_" + str(v)]
    for u,v,a in G.in_edges(n, data=True):
        inDegreeThisNode += gurobiVars[str(u) + "_" + str(v)]


    m.addConstr(inDegreeThisNode == outDegreeThisNode)

In [111]:
#now add constraints. A eulerian tour requires (and will always exist if) 
# all nodes are even 

#to do this, iterate over all nodes. Create a constraint that the total
# number of paths for all edges going into or out of that node must be even.

#note that Gurobi doesn't allow modulo operator
#  so to solve that, declare a dedicated int variable for each
#constraint, and then express each constraint in the form X + Y = 2z (for evenness)
#that thisNodeEvenOddVar doesn't matter for the outcome, it will just constrain to even

for n in G.nodes():
    constraintForThisNode = LinExpr()
    thisNodeEvenOddVar = m.addVar(vtype=GRB.INTEGER, name="EvennessForNode" + str(n))
    for u,v,a in G.edges(n, data=True):
        constraintForThisNode += gurobiVars[str(u) + "_" + str(v)]
    m.addConstr(constraintForThisNode == (2*thisNodeEvenOddVar))

In [112]:
# Subtours

#The first time I ran this, I got this output:
#[(34, 35, {'weight': 0.35, 'required': 1, 'forwardTrips': 1.0, 'reverseTrips': 1.0})
# , (34, 31, {'weight': 0.2, 'required': 0, 'forwardTrips': -0.0, 'reverseTrips': -0.0})]
#this is for the bear rocks loop at the top. going from 34-35 and back makes
# a small complete loop, so it satisfies all the requirements, but isn't 
# connected to anything else. So the entire output doesn't make a single tour 

# There is a proper way to do this documented here: 
# https://colab.research.google.com/github/Gurobi/modeling-examples/blob/master/traveling_salesman/tsp.ipynb#scrollTo=MuqBd8AdLYAC
# but it's complicated. I'm going to first try manually eliminating subtours
# as I find them.


eliminateBearLoopSubtour = LinExpr()
eliminateBearLoopSubtour += gurobiVars['34_35']
eliminateBearLoopSubtour += gurobiVars['35_34']
eliminateBearLoopSubtour += gurobiVars['34_31']
eliminateBearLoopSubtour += gurobiVars['31_34']

m.addConstr(eliminateBearLoopSubtour >= 3)



<gurobi.Constr *Awaiting Model Update*>

In [113]:
#do the thing
m.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: AMD Ryzen 7 9700X 8-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 118 rows, 141 columns and 439 nonzeros (Min)
Model fingerprint: 0x28bafddb
Model has 104 linear objective coefficients
Variable types: 0 continuous, 141 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [1e-01, 3e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 3e+00]

Presolve removed 6 rows and 6 columns
Presolve time: 0.00s
Presolved: 112 rows, 135 columns, 419 nonzeros
Variable types: 0 continuous, 135 integer (0 binary)

Root relaxation: objective 3.045000e+01, 127 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Tim

In [114]:
m.display()

Minimize
3.2 0_7 + 3.1 0_1 + 2.2 0_4 + 0.5 7_10 + 0.4 7_0 + 0.4 7_4 + 1.2 10_12 + 0.4 10_7
+ 0.4 10_5 + 1.3 18_25 + 1.9 18_20 + 0.4 18_17 + 0.1 25_24 + 0.4 25_18 + 0.4 25_28
+ 1.3 5_10 + 0.7 5_9 + 0.4 5_3 + 1.5 12_14 + 0.4 12_10 + 0.4 12_4 + 0.3 19_21
+ 0.4 19_11 + 0.4 19_13 + 1.3 21_22 + 0.5 21_23 + 0.4 21_19 + 0.9 22_17 + 1.4 22_23
+ 0.4 22_21 + 0.3 17_18 + 0.4 17_22 + 0.4 17_13 + 1.4 20_27 + 0.4 20_18 + 0.4 20_15
+ 1.3 3_8 + 1.5 3_5 + 0.4 3_2 + 0.6 8_11 + 1.9 8_9 + 0.4 8_3 + 2.5 11_19 + 0.4 11_8
+ 0.4 11_6 + 0.7 1_2 + 0.4 1_0 + 2_3 + 1.4 2_6 + 0.4 2_1 + 1.4 9_13 + 0.4 9_5 + 0.4 9_8
+ 1.1 13_17 + 2.5 13_19 + 0.4 13_9 + 0.7 27_29 + 0.8 27_31 + 0.4 27_20 + 0.7 29_32
+ 0.4 29_27 + 0.4 29_24 + 1.4 24_30 + 1.1 24_29 + 0.4 24_25 + 0.2 30_33 + 1.3 30_32
+ 0.4 30_24 + 1.2 33_36 + 0.4 33_30 + 0.4 33_28 + 0.4 36_33 + 0.4 36_26 + 0.4 32_29
+ 0.4 32_30 + 0.4 32_31 + 1.1 31_32 + 0.2 31_34 + 0.4 31_27 + 1.2 28_33 + 0.9 28_25
+ 0.4 28_26 + 0.9 23_26 + 0.4 23_21 + 0.4 23_22 + 1.7 26_36 + 1.6 26_28 +

/tmp/ipykernel_3991869/137993151.py:1: DeprecationWarning: Model.display() is deprecated
  m.display()


In [115]:
#print the results, sorted by variable name
vList = m.getVars()
vList.sort(key=lambda var: int(var.varName[:var.varName.find("_",0)]) if var.varName.find("_",0) > 0 else 1000 )


for v in vList:
    print('%s %g' % (v.varName, v.x))

print('Obj: %g' % m.objVal)

0_7 1
0_1 -0
0_4 1
1_2 1
1_0 1
2_3 1
2_6 1
2_1 2
3_8 1
3_5 -0
3_2 1
4_7 0
4_12 2
4_0 -0
5_10 1
5_9 0
5_3 1
6_11 -0
6_2 2
7_10 0
7_0 1
7_4 1
8_11 2
8_9 -0
8_3 0
9_13 -0
9_5 1
9_8 1
10_12 -0
10_7 1
10_5 1
11_19 1
11_8 -0
11_6 1
12_14 1
12_10 1
12_4 -0
13_17 0
13_19 -0
13_9 2
14_16 2
14_15 0
14_12 -0
15_20 2
15_14 0
15_16 -0
16_15 1
16_14 1
17_18 1
17_22 0
17_13 1
18_25 1
18_20 -0
18_17 1
19_21 1
19_11 0
19_13 1
20_27 -0
20_18 1
20_15 1
21_22 0
21_23 1
21_19 1
22_17 1
22_23 -0
22_21 1
23_26 -0
23_21 -0
23_22 2
24_30 -0
24_29 1
24_25 1
25_24 0
25_18 0
25_28 2
26_36 1
26_28 -0
26_23 1
27_29 1
27_31 1
27_20 0
28_33 1
28_25 0
28_26 1
29_32 2
29_27 -0
29_24 0
30_33 0
30_32 -0
30_24 2
31_32 0
31_34 0
31_27 2
32_29 -0
32_30 1
32_31 1
33_36 1
33_30 1
33_28 0
34_35 2
34_31 0
35_34 2
36_33 1
36_26 1
EvennessForNode0 1
EvennessForNode7 1
EvennessForNode10 1
EvennessForNode18 1
EvennessForNode25 1
EvennessForNode5 1
EvennessForNode12 1
EvennessForNode19 1
EvennessForNode21 1
EvennessForNode22 1
Evenn

In [116]:
#so the answer is that if you do it all in one trip, the total mileage would be
# 63.1. The total mileage of the trails by themselves is 48.5, so the complete
#tour requires 14.6 miles of doubling back/walking on roads

In [117]:
#update the graph to add the num times to each node

for u,v,a in G.edges(data=True):
    for var in vList:
        if(var.varName == f'{u}_{v}'):
            a['trips'] = var.x



In [118]:
#now build the tour
#start with node 31 - the main parking lot.

lastNode = -1
currentNode = 31

def tripsRemaining():
    remaining = 0
    for u,v,a in G.edges(data=True):
        remaining += a['trips']
    return remaining

def selectNextEdge():
    #first, try to find an edge that doesn't just double back
    for u, v, a in G.edges(currentNode,data=True):
        if (a['trips'] >= 1 and v != lastNode):
            a['trips'] -= 1
            return(u,v)
    #if doubling back is our only option, do that
    for u, v, a in G.edges(currentNode,data=True):
        if (a['trips'] >= 1):
            a['trips'] -= 1
            return(u,v)
    print("error: should have been able to get an edge from this node")


while tripsRemaining():
    #select an edge that comes out of this node that has trips remaining
    print(f"Node: {currentNode}")
    u, v = selectNextEdge()
    print(f"take edge from {u} to {v}") 
    lastNode = currentNode
    currentNode = v






Node: 31
take edge from 31 to 27
Node: 27
take edge from 27 to 29
Node: 29
take edge from 29 to 32
Node: 32
take edge from 32 to 30
Node: 30
take edge from 30 to 24
Node: 24
take edge from 24 to 29
Node: 29
take edge from 29 to 32
Node: 32
take edge from 32 to 31
Node: 31
take edge from 31 to 27
Node: 27
take edge from 27 to 31
Node: 31
error: should have been able to get an edge from this node


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
print(G.out_edges(30,data=True))
print(G.in_edges(30,data=True))

[(30, 33, {'weight': 0.2, 'required': 0, 'trips': 0.0}), (30, 32, {'weight': 1.3, 'required': 0, 'trips': -0.0}), (30, 24, {'weight': 1.4, 'required': 0, 'trips': -0.0})]
[(24, 30, {'weight': 1.4, 'required': 0, 'trips': -0.0}), (33, 30, {'weight': 0.2, 'required': 0, 'trips': 0.0}), (32, 30, {'weight': 1.3, 'required': 0, 'trips': -0.0})]
